In [4]:
import numpy as np

# Step 0 — Inputs you assume you already have

## For each subject:

-   A list of that subject’s electrode locations (3D MNI coordinates)

## For each session:

-   A matrix of voltage time series: timepoints × electrodes

-   A list of all locations you want to index the final matrix by

In the paper, this is “all electrode locations from everyone pooled together” (union across subjects).

A smoothing width value (lambda), set to 20 in the paper.

In [6]:
# Each subjets electrode locations, registered to MNI space, in mm
aa_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/aa_xslocs_registered_mm.npy")
ap_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/ap_xslocs_registered_mm.npy")
ca_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/ca_xslocs_registered_mm.npy")
de_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/de_xslocs_registered_mm.npy")
fp_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/fp_xslocs_registered_mm.npy")
ha_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/ha_xslocs_registered_mm.npy")
ja_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/ja_xslocs_registered_mm.npy")
jm_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/jm_xslocs_registered_mm.npy")
jt_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/jt_xslocs_registered_mm.npy")
mv_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/mv_xslocs_registered_mm.npy")
rn_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/rn_xslocs_registered_mm.npy")
rr_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/rr_xslocs_registered_mm.npy")
wc_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/wc_xslocs_registered_mm.npy")
zt_locs = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs/zt_xslocs_registered_mm.npy")

### The order of the electrode numbers in the data should match the order of the electrode locations, so that you can compute distances between them and apply the weights correctly.

In [ ]:
# each subjects preprocessed voltage time series, in microvolts, per electrode

# aa_data = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/preprocessed_outputs/aa_xsdata_preprocessed.npy") ###

In [5]:
# All subjects' electrode locations in mni

all_electrodes = np.load("/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/all_electrodes_mni_locs.npy")


# Step 1 — For each subject, compute “how correlated are that subject’s electrodes with each other?”

## Output: per subject, a square matrix: electrode × electrode correlation values.

FOR each subject:

    # 1A) compute a correlation matrix for each session
    session_correlation_matrices = []

    FOR each session:

        # voltages: timepoints × electrodes
        voltages = subject.session[session].voltages

        # compute Pearson correlation across electrodes (columns)
        corr_matrix = pearson_correlation_between_columns(voltages)

        session_correlation_matrices.append(corr_matrix)

    # 1B) average correlations across sessions using Fisher-z trick
    # (they do z-transform, average, then inverse transform) :contentReference[oaicite:3]{index=3}

    z_matrices = apply_fisher_z_to_each_matrix(session_correlation_matrices)
    mean_z = elementwise_average(z_matrices)
    subject_average_correlation_matrix = inverse_fisher_z(mean_z)

    save subject_average_correlation_matrix


# Step 2 — For each subject, compute “how much does each of their electrodes inform each pooled location?”

## Output: for each subject, a table of weights:
pooled_location × subject_electrode

FOR each subject:

    subject_electrode_locations = subject.electrode_locations  # list of 3D coords
    pooled_locations = all_locations_across_all_subjects       # list of 3D coords

    # Make a weight table:
    # rows = pooled locations
    # cols = this subject's electrodes
    weights = zeros(num_pooled_locations, num_subject_electrodes)

    FOR each pooled_location_index, pooled_coord in pooled_locations:

        FOR each electrode_index, electrode_coord in subject_electrode_locations:

            distance_squared = squared_euclidean_distance(pooled_coord, electrode_coord)

            # Gaussian falloff (Eq. 1) :contentReference[oaicite:5]{index=5}
            weight = exp( -distance_squared / lambda )

            weights[pooled_location_index, electrode_index] = weight

    save weights


# Step 3A — Subject-specific “full” matrix over pooled locations

## Goal: For a given subject, estimate correlation between any two pooled locations using:

that subject’s measured electrode correlations (Step 1)

the distance-based weights (Step 2)

FOR each subject:

    electrode_corr = subject_average_correlation_matrix   # electrodes × electrodes
    weights = subject_weights_table                       # pooled_locations × electrodes

    # subject_full_matrix is pooled_locations × pooled_locations
    subject_full_matrix = zeros(num_pooled_locations, num_pooled_locations)

    FOR each pooled_location_A in pooled_locations:

        FOR each pooled_location_B in pooled_locations:

            numerator_sum = 0
            denominator_sum = 0

            FOR each electrode_i in subject_electrodes:
                FOR each electrode_j in subject_electrodes:

                    # how much electrode i "represents" pooled_location_A
                    weight_A_i = weights[pooled_location_A, electrode_i]

                    # how much electrode j "represents" pooled_location_B
                    weight_B_j = weights[pooled_location_B, electrode_j]

                    pair_weight = weight_A_i * weight_B_j

                    # they Fisher-z the electrode correlations before averaging :contentReference[oaicite:7]{index=7}
                    corr_ij_z = fisher_z(electrode_corr[electrode_i, electrode_j])

                    numerator_sum   += pair_weight * corr_ij_z
                    denominator_sum += pair_weight

            # weighted average, then inverse Fisher-z (Eq. 6 idea) :contentReference[oaicite:8]{index=8}
            mean_z = numerator_sum / denominator_sum
            subject_full_matrix[pooled_location_A, pooled_location_B] = inverse_fisher_z(mean_z)

    save subject_full_matrix


# Step 3B — Average the subject-specific full matrices to get the across-subject matrix

collect all subject_full_matrix objects: one per subject
all_subject_full_matrices = [...]

Fisher-z them, average them elementwise, inverse Fisher-z (Eq. 7) :contentReference[oaicite:9]{index=9}
z_mats = fisher_z(all_subject_full_matrices)
mean_z = elementwise_average(z_mats)
across_subject_correlation_matrix = inverse_fisher_z(mean_z)
